In [ ]:
!pip install unsloth

  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
ERROR: Operation cancelled by user
^C


In [ ]:
pip install -U transformers accelerate peft trl datasets bitsandbytes evaluate scikit-learn

In [ ]:
!pip install --upgrade torchao

In [ ]:
!pip uninstall -y torchaudio

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

msgs = [{"role": "user", "content": "I want a 32 inch TV under $300."}]
inputs = tokenizer.apply_chat_template(
    msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)

out = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(out[0], skip_special_tokens=True))

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

user
I want a 32 inch TV under $300.
assistant
<think>

</think>

Finding a **32-inch TV under $300** is actually quite achievable, but it requires a specific strategy because 32-inch TVs are generally priced higher than 27-inch ones. You will likely need to look for **budget


In [ ]:
#!/usr/bin/env python3
"""
LoRA fine-tune Qwen3.5-0.8B using Unsloth for ultra-fast training on T4.

Expects train.jsonl / val.jsonl produced by generate_synthetic_data.py, each line:
  {"query": "...", "main_category": "...", "target": "{\"main_category\": \"...\"}"}
"""

# ---- 1. Fix for the Unsloth Qwen3.5 Compiler Bug ----
# Force-injecting this into the environment BEFORE any unsloth imports happen
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel

import argparse
import re
import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

SYSTEM_PROMPT = (
    "You convert a shopping request into a structured product query. "
    "Respond with only a JSON object containing the field main_category."
)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model-name", default="unsloth/Qwen3.5-0.8B")
    ap.add_argument("--train", default="train.jsonl")
    ap.add_argument("--val", default="val.jsonl")
    ap.add_argument("--output-dir", default="./qwen-category-lora")
    ap.add_argument("--epochs", type=float, default=1.0)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--grad-accum", type=int, default=1)
    ap.add_argument("--max-seq-length", type=int, default=256)
    ap.add_argument("--lora-r", type=int, default=16)
    ap.add_argument("--lora-alpha", type=int, default=32)
    ap.add_argument("--seed", type=int, default=42)
    args, _unknown = ap.parse_known_args()

    # ---- Unsloth Model & Tokenizer Loading ----
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=args.model_name,
        max_seq_length=args.max_seq_length,
        load_in_4bit=True,
    )

    # ---- Unsloth LoRA Patching ----
    model = FastLanguageModel.get_peft_model(
        model,
        r=args.lora_r,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_alpha=args.lora_alpha,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=args.seed,
    )

    # ---- data ----
    ds = load_dataset(
        "json", data_files={"train": args.train, "validation": args.val}
    )

    def formatting_prompts_func(examples):
        queries = examples["query"]
        targets = examples["target"]
        texts = []
        for query, target in zip(queries, targets):
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": query},
                {"role": "assistant", "content": target},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
        return texts

    sft_config = SFTConfig(
        output_dir=args.output_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        max_seq_length=args.max_seq_length,
        packing=False,
        dataset_text_field="text",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        seed=args.seed,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=ds["train"],
        eval_dataset=ds["validation"],
        formatting_func=formatting_prompts_func,
    )

    trainer.train()

    # ---- save adapter only ----
    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print(f"Saved LoRA adapter + tokenizer to {args.output_dir}")

    # ---- quick sanity check ----
    print("\n--- Sample generations on validation set ---")
    FastLanguageModel.for_inference(model)
    try:
        for ex in ds["validation"].select(range(min(5, len(ds["validation"])))):
            msgs = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": ex["query"]},
            ]
            inputs = tokenizer.apply_chat_template(
                msgs,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            ).to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=40, use_cache=True
                )
            pred = tokenizer.decode(
                out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
            )
            print(f"query: {ex['query']}")
            print(f"  true: {ex['target']}")
            print(f"  pred: {pred.strip()}")
    except Exception as e:
        print(f"(Sanity-check generation skipped due to: {e})")


if __name__ == "__main__":
    main()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1427: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen3_5 patching. Transformers: 5.13.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Unsloth: Switching to float32 training since model cannot work with float16


...

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,500 | Num Epochs = 1 | Total steps = 657
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 6,389,760 of 859,375,680 (0.74% trained)


Epoch,Training Loss,Validation Loss
1,1.533560,1.507470


Unsloth: Restored added_tokens_decoder metadata in ./qwen-category-lora/tokenizer_config.json.


Saved LoRA adapter + tokenizer to ./qwen-category-lora

--- Sample generations on validation set ---
(Sanity-check generation skipped due to: 'str' object has no attribute 'to')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ---- 1. Save LoRA Adapters locally ----
    # Instead of args.output_dir, we can use the default string or define it
local_output_dir = "./qwen-category-lora"
model.save_pretrained(local_output_dir)
tokenizer.save_pretrained(local_output_dir)
print(f"Saved LoRA adapter + tokenizer locally to {local_output_dir}")

    # ---- 2. Save permanently to Google Drive ----
gdrive_output_dir = "/content/drive/MyDrive/qwen-category-lora"

    # Ensure the directory exists (optional, but good practice)
os.makedirs(gdrive_output_dir, exist_ok=True)

    # Save the lightweight adapters to Drive
model.save_pretrained(gdrive_output_dir)
tokenizer.save_pretrained(gdrive_output_dir)
print(f"Saved LoRA adapter + tokenizer to Google Drive: {gdrive_output_dir}")

    # ---- 3. Save Merged Model (Crucial for evaluation later!) ----
try:
    print("Saving merged 16bit model to Google Drive for easier evaluation...")
    model.save_pretrained_merged(
            gdrive_output_dir + "-merged",
            tokenizer,
            save_method="merged_16bit"
        )
    print("Merged 16bit model saved successfully!")
except Exception as e:
    print(f"Failed to save merged model: {e}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved LoRA adapter + tokenizer locally to ./qwen-category-lora


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved LoRA adapter + tokenizer to Google Drive: /content/drive/MyDrive/qwen-category-lora
Saving merged 16bit model to Google Drive for easier evaluation...
Failed to save merged model: 'Qwen3_5ForCausalLM' object has no attribute 'save_pretrained_merged'


In [ ]:
import os

gdrive_output_dir = "/content/drive/MyDrive/qwen-category-lora"

try:
    print("Saving merged 16bit model to Google Drive for easier evaluation...")

    # Correct Unsloth syntax for merging and saving locally/Drive
    model.save_pretrained(
        save_directory = gdrive_output_dir + "-merged",
        tokenizer = tokenizer,
        save_method = "merged_16bit",
    )
    print("Merged 16bit model saved successfully!")
except Exception as e:
    print(f"Failed to save merged model: {e}")

Saving merged 16bit model to Google Drive for easier evaluation...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged 16bit model saved successfully!


In [ ]:
#!/usr/bin/env python3
"""
Evaluate the FINE-TUNED Qwen3.5 model using the saved LoRA adapter on a test set.
Extracts: Accuracy, Macro F1, and a Confusion Matrix.
"""

import os
# Fix for the Unsloth Qwen3.5 Compiler Bug
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

import json
import re
import argparse
import torch
from tqdm import tqdm
from datasets import load_dataset
from unsloth import FastLanguageModel
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SYSTEM_PROMPT = (
    "You convert a shopping request into a structured product query. "
    "Respond with only a JSON object containing the field main_category."
)


def extract_category(text):
    """
    Parses the model's output to find the 'main_category' value.
    Tries strict JSON parsing first, then falls back to regex.
    """
    text = text.strip()
    # 1. Try direct JSON parsing
    try:
        data = json.loads(text)
        if "main_category" in data:
            return str(data["main_category"]).strip()
    except Exception:
        pass

    # 2. Fallback: Regex extraction in case of minor formatting anomalies
    try:
        match = re.search(r'"main_category"\s*:\s*"([^"]+)"', text)
        if match:
            return match.group(1).strip()
    except Exception:
        pass

    return "UNKNOWN"

def main():
    ap = argparse.ArgumentParser()
    # Point this to your saved adapter directory from the fine-tuning script
    ap.add_argument("--adapter-dir", default="./qwen-category-lora")
    ap.add_argument("--test-file", default="test.jsonl")
    ap.add_argument("--max-seq-length", type=int, default=256)
    args, _ = ap.parse_known_args()

    # ---- 1. Load Fine-Tuned Model (Base + LoRA Adapter) ----
    print(f"Loading fine-tuned adapter from: {args.adapter_dir}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=args.adapter_dir, # Loading the adapter directory auto-merges for inference
        max_seq_length=args.max_seq_length,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    # ---- 2. Load Test Dataset ----
    print(f"Loading test data from: {args.test_file}")
    ds = load_dataset("json", data_files={"test": args.test_file})["test"]

    y_true = []
    y_pred = []


# ---- 3. Inference Loop ----
    print("\nRunning evaluation on test data...")
    for item in tqdm(ds):
        query = item["query"]

        # Get true category
        true_cat = item.get("main_category", None)
        if not true_cat and "target" in item:
            true_cat = extract_category(item["target"])

        if not true_cat:
            continue

        # Format input prompt exactly like the training template
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ]

        # 1. Get raw prompt string
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        # 2. Tokenize into raw integer tensor lists
        #inputs = tokenizer(prompt_text, return_tensors="pt")
        inputs = tokenizer(text=prompt_text, return_tensors="pt")
        input_ids = inputs["input_ids"].to(model.device)
        attention_mask = inputs["attention_mask"].to(model.device)

        with torch.no_grad():
            # Added images=None to explicitly shut off the transformers image_utils scanner
            out = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=40,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
                images=None, # <--- CRITICAL FIX
            )

        # Isolate the newly generated tokens
        generated_tokens = out[0][input_ids.shape[1]:]
        raw_pred = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        # Clean and extract category
        pred_cat = extract_category(raw_pred)

        y_true.append(str(true_cat).strip())
        y_pred.append(pred_cat)

    # ---- 4. Metrics Reporting ----
    print("\n" + "="*50)
    print(" FINE-TUNED MODEL EVALUATION RESULTS ")
    print("="*50)

    unique_labels = sorted(list(set(y_true)))
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"Total Test Samples: {len(y_true)} (15 categories x 80 samples)")
    print(f"Accuracy:           {accuracy:.4f}")
    print(f"Macro F1-Score:     {macro_f1:.4f}")
    print("\n--- Detailed Classification Report ---")
    print(classification_report(y_true, y_pred, labels=unique_labels, zero_division=0))

    print("\n--- Confusion Matrix ---")
    cm = confusion_matrix(y_true, y_pred, labels=unique_labels)

    # Render scannable console matrix table
    print(f"{'True \\ Pred':<25}", end="")
    for label in unique_labels:
        print(f"| {label[:8]:<8}", end="")
    print("\n" + "-" * (25 + 11 * len(unique_labels)))

    for i, label in enumerate(unique_labels):
        print(f"{label[:23]:<25}", end="")
        for j in range(len(unique_labels)):
            print(f"| {cm[i][j]:<8}", end="")
        print()

if __name__ == "__main__":
    main()

Loading fine-tuned adapter from: ./qwen-category-lora
==((====))==  Unsloth 2026.7.2: Fast Qwen3_5 patching. Transformers: 5.13.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Loading test data from: test.jsonl

Running evaluation on test data...


100%|██████████| 1200/1200 [28:19<00:00,  1.42s/it]


 FINE-TUNED MODEL EVALUATION RESULTS 
Total Test Samples: 1200 (15 categories x 80 samples)
Accuracy:           0.6400
Macro F1-Score:     0.6392

--- Detailed Classification Report ---
                               precision    recall  f1-score   support

Arts, Crafts & Party Supplies       0.54      0.85      0.66        80
                   Automotive       0.83      0.68      0.74        80
                Baby Products       0.33      0.91      0.48        80
       Beauty & Personal Care       0.82      0.86      0.84        80
      Electronics & Computers       0.86      0.64      0.73        80
     Fashion, Shoes & Luggage       0.73      0.66      0.69        80
           Health & Household       0.67      0.50      0.57        80
               Home & Kitchen       0.58      0.51      0.54        80
      Industrial & Scientific       0.89      0.10      0.18        80
                 Pet Supplies       0.95      0.76      0.85        80
                   Smart Home  